# How the scoring engine works

This notebook follows one fictional CV and job description through the real scoring code, one step at a time.

```text
Job description ──> 1. Find requirements
CV ───────────────> 2. Read the evidence
                    3. Grade each requirement
                    4. Calculate the score
                    5. Build the full result
```

Run it from the repo root with:

```sh
uv run --with jupyter jupyter lab notebooks/scoring_engine.ipynb
```

## Setup

`show` prints a simple table. The date is fixed so that time in a current role, and so the results, stay the same on every run.

In [1]:
from datetime import date

from IPython.display import Markdown, display

from mycareerai.module.scoring import analyze, parse_job
from mycareerai.module.scoring.match import CREDIT, _find_line, match_requirement, read_cv
from mycareerai.module.scoring.score import FIT_WEIGHTS, score_fit, score_quality

TODAY = date(2026, 11, 1)


def show(columns, rows):
    clean = lambda value: "–" if value is None else str(value).replace("|", "\\|")
    lines = [" | ".join(columns), " | ".join("---" for _ in columns)]
    lines += [" | ".join(clean(value) for value in row) for row in rows]
    display(Markdown("\n".join(f"| {line} |" for line in lines)))

## The example

Both documents are made up for this notebook.

In [2]:
JOB = """Northwind is hiring a Data Analyst.

Responsibilities:
- Build dashboards in Tableau or Power BI

Requirements:
- 3+ years of experience in analytics
- Strong SQL and Python skills
- Experience with Docker
- Data Visualization
- Bachelor's degree
- Knowledge of retail pricing models

Nice to have:
- Snowflake
- Excellent communication
"""

CV = """SUMMARY
Analyst who turns sales data into decisions.

EXPERIENCE
Analyst, Harbourline (Jan 2024 – Dec 2025)
- Wrote SQL queries that cut reporting time by 30%
- Built Power BI reports for 12 regional teams
- Modelled discount pricing for 40 retail stores

EDUCATION
BSc Business Analytics, Example University, 2023

SKILLS
SQL, Python, Power BI, Communication
"""

## Step 1: Find the requirements

`parse_job` turns the job description into a checklist.

- **Importance** comes from the headings and wording. Soft skills are always preferred.
- **Tableau or Power BI** is one requirement, because either skill satisfies it.
- **A line with no known skill** has an empty Known skills column. Without a model, it can't be assessed.

In [3]:
job = parse_job(JOB)

show(
    ["Requirement", "Kind", "Importance", "Known skills"],
    [(r.text, r.kind, r.importance, ", ".join(r.skills)) for r in job.requirements],
)

| Requirement | Kind | Importance | Known skills |
| --- | --- | --- | --- |
| Tableau or Power BI | skill | preferred | Tableau, Power BI |
| SQL | skill | must | SQL |
| Python | skill | must | Python |
| Docker | skill | must | Docker |
| Data Visualization | skill | must | Data Visualization |
| Knowledge of retail pricing models | skill | must |  |
| Snowflake | skill | preferred | Snowflake |
| Communication | skill | preferred | Communication |
| 3+ years of experience in analytics | experience | must |  |
| Bachelor's degree | education | must |  |

## Step 2: Read the CV

`read_cv` collects what the grading step needs:

- **Evidence lines** are the work and project lines, which count as proof of use.
- **Months of experience** come from the dated roles, and overlapping roles aren't counted twice.
- **Degree level** runs from 1 (diploma) to 4 (PhD).

In [4]:
cv = read_cv(CV, TODAY)

print("Months of experience:", cv.months_of_experience)
print("Degree level:", cv.degree_level)
print("Evidence lines:")
for line in cv.evidence:
    print("  -", line)

Months of experience: 24
Degree level: 2
Evidence lines:
  - Analyst, Harbourline (Jan 2024 – Dec 2025)
  - Wrote SQL queries that cut reporting time by 30%
  - Built Power BI reports for 12 regional teams
  - Modelled discount pricing for 40 retail stores


## Step 3: Grade each requirement

Each requirement earns a tier, and each tier earns a fixed credit:

In [5]:
meaning = {
    "demonstrated": "In a work or project line",
    "implied": "A model found a similar work or project line",
    "listed": "Elsewhere in the CV, usually under Skills",
    "related": "A similar skill was found",
    "missing": "Not found",
}
show(["Tier", "Credit", "Meaning"], [(tier, credit, meaning[tier]) for tier, credit in CREDIT.items()])

| Tier | Credit | Meaning |
| --- | --- | --- |
| demonstrated | 1.0 | In a work or project line |
| implied | 0.8 | A model found a similar work or project line |
| listed | 0.6 | Elsewhere in the CV, usually under Skills |
| related | 0.4 | A similar skill was found |
| missing | 0.0 | Not found |

`_find_line` does the core search: it returns the first line that contains a skill. Searching the evidence lines first is what separates using a skill from only listing it.

In [6]:
all_lines = CV.splitlines()

show(
    ["Skill", "Found in evidence lines", "Found anywhere in the CV"],
    [(skill, _find_line(skill, cv.evidence), _find_line(skill, all_lines)) for skill in ["SQL", "Python", "Docker"]],
)

| Skill | Found in evidence lines | Found anywhere in the CV |
| --- | --- | --- |
| SQL | Wrote SQL queries that cut reporting time by 30% | - Wrote SQL queries that cut reporting time by 30% |
| Python | – | SQL, Python, Power BI, Communication |
| Docker | – | – |

Now every requirement is graded. Two tiers aren't in the table above:

- **partial** means some of the required years, or a lower qualification.
- **unassessed** means it needs a model, so it's left out of the score.

In [7]:
matches = [match_requirement(requirement, cv) for requirement in job.requirements]

show(
    ["Requirement", "Importance", "Tier", "Credit", "Evidence"],
    [
        (m.requirement.text, m.requirement.importance, m.tier, None if m.credit is None else round(m.credit, 2), m.evidence)
        for m in matches
    ],
)

| Requirement | Importance | Tier | Credit | Evidence |
| --- | --- | --- | --- | --- |
| Tableau or Power BI | preferred | demonstrated | 1.0 | Built Power BI reports for 12 regional teams |
| SQL | must | demonstrated | 1.0 | Wrote SQL queries that cut reporting time by 30% |
| Python | must | listed | 0.6 | SQL, Python, Power BI, Communication |
| Docker | must | missing | 0.0 | – |
| Data Visualization | must | related | 0.4 | Built Power BI reports for 12 regional teams |
| Knowledge of retail pricing models | must | unassessed | – | – |
| Snowflake | preferred | missing | 0.0 | – |
| Communication | preferred | listed | 0.6 | SQL, Python, Power BI, Communication |
| 3+ years of experience in analytics | must | partial | 0.67 | About 2.0 years across dated roles |
| Bachelor's degree | must | demonstrated | 1.0 | – |

## Step 4: Calculate the score

The starting weights are below. If a job has no requirements in a category, it's dropped and the others are scaled up.

In [8]:
show(["Category", "Starting weight"], FIT_WEIGHTS.items())

| Category | Starting weight |
| --- | --- |
| Must-have skills | 65 |
| Preferred skills | 15 |
| Experience | 15 |
| Education | 5 |

Each category's score is the average credit of its requirements, multiplied by 100. Displayed weights and scores are rounded, so the points can differ slightly from the final score.

In [9]:
fit = score_fit(matches)

show(
    ["Category", "Scaled weight", "Score", "Points added", "Detail"],
    [(p.label, f"{p.weight}%", p.score, round(p.weight * p.score / 100, 1), p.detail) for p in fit.parts],
)
print("Final score:", fit.overall)
print("Limit applied:", fit.cap)

| Category | Scaled weight | Score | Points added | Detail |
| --- | --- | --- | --- | --- |
| Must-have skills | 65% | 50 | 32.5 | 1 of 4 shown in your work or projects |
| Preferred skills | 15% | 53 | 8.0 | 1 of 3 shown in your work or projects |
| Experience | 15% | 67 | 10.1 | About 2.0 years across dated roles |
| Education | 5% | 100 | 5.0 | Meets the stated level |

Final score: 56
Limit applied: None


### Why limits exist

An average can hide a dealbreaker. Here the CV is strong everywhere except one must-have, Docker. The average is high, but a missing must-have limits the score to 79, so it can't reach "Strong".

In [10]:
CAP_JOB = """Requirements:
- SQL
- Python
- Power BI
- Excel
- Tableau
- Docker
"""
CAP_CV = """EXPERIENCE
Analyst, Harbourline (Jan 2020 – Dec 2025)
- Used SQL, Python, Power BI, Excel and Tableau in weekly reporting
"""

capped = analyze("Analyst", "Northwind", CAP_JOB, CAP_CV, today=TODAY)
print("Must-have coverage:", capped["parts"][0]["score"])
print("Final score:", capped["overall"])
print("Limit applied:", capped["cap"])

Must-have coverage: 83
Final score: 79
Limit applied: 79


### CV quality is separate

`score_quality` rates how the CV is written. It doesn't change the match score, because a tidy CV doesn't make someone a better fit for the job.

In [11]:
quality = score_quality(CV)

print("CV quality:", quality.score)
show(["Part", "Weight", "Score", "Detail"], [(p.label, f"{p.weight}%", p.score, p.detail) for p in quality.parts])

CV quality: 84


| Part | Weight | Score | Detail |
| --- | --- | --- | --- |
| Impact | 60% | 100 | 3 of 3 bullet points quantified · 0 weak phrases |
| Structure | 40% | 60 | Missing: Projects or certifications |

## Step 5: The full result

`analyze` runs every step above. This is what the Score page receives from the API, shown here without the input text and the methodology.

In [12]:
result = analyze("Data Analyst", "Northwind", JOB, CV, today=TODAY)

print(f"Score: {result['overall']} ({result['bandLabel']})")
print("Confidence:", result["confidence"]["level"])
for reason in result["confidence"]["reasons"]:
    print("  -", reason)
print("Matched:", ", ".join(result["matched"]))
print("Missing:", ", ".join(result["missing"]))
print("Engine version:", result["engineVersion"])
print()
print("Recommendations:")
for item in result["recommendations"]:
    print(f"  [{item['severity']}] {item['text']}")

Score: 56 (Fair match)
Confidence: medium
  - 1 requirement could not be assessed and was left out of the score.
Matched: Tableau or Power BI, SQL, Python, Communication
Missing: Docker, Data Visualization, Snowflake
Engine version: rules-1

Recommendations:
  [high] Required but not shown: **Docker and Data Visualization**. Add evidence if you have it.
  [high] Mentioned but not shown in use: **Python**. Add it to a work or project bullet.
  [med] Useful but not mentioned: **Snowflake**. Add it if you have it.
  [low] Consider adding a section for projects or certifications.


## Does the score behave sensibly?

Each scenario changes one thing. Listing a skill should earn less than showing it in work, and removing evidence should lower the score.

In [13]:
FRONTEND_JOB = """Requirements:
- React
- TypeScript
- CSS
- Figma
"""


def score(cv=CV, job=JOB):
    outcome = analyze("Data Analyst", "Northwind", job, cv, today=TODAY)
    return outcome["overall"], outcome["bandLabel"]


scenarios = {
    "Original CV": score(),
    "Docker added to the Skills list only": score(CV.replace("SQL, Python, Power BI, Communication", "SQL, Python, Power BI, Communication, Docker")),
    "Docker added to a work bullet": score(CV.replace("- Built Power BI reports", "- Deployed reports in Docker\n- Built Power BI reports")),
    "SQL bullet deleted": score(CV.replace("- Wrote SQL queries that cut reporting time by 30%\n", "")),
    "Same CV, unrelated job": score(job=FRONTEND_JOB),
}

show(["Scenario", "Score", "Band"], [(name, *outcome) for name, outcome in scenarios.items()])

| Scenario | Score | Band |
| --- | --- | --- |
| Original CV | 56 | Fair match |
| Docker added to the Skills list only | 65 | Good match |
| Docker added to a work bullet | 72 | Good match |
| SQL bullet deleted | 49 | Fair match |
| Same CV, unrelated job | 0 | Weak match |

## Where BERT fits

Without a model, the free-text pricing requirement stays `unassessed`. A model can compare it with the CV's bullets.

**The encoder below is a fake.** It gives texts about pricing the same vector and every other text a similarity of 0, so this shows the routing and says nothing about real BERT accuracy. No real model is connected yet, and the similarity thresholds are still placeholders.

In [14]:
class FakeEncoder:
    version = "fake-1"

    def encode(self, texts):
        return [[1.0, 0.0] if "pricing" in text.lower() else [0.0, 0.0] for text in texts]


without_model = analyze("Data Analyst", "Northwind", JOB, CV, today=TODAY)
with_model = analyze("Data Analyst", "Northwind", JOB, CV, today=TODAY, encoder=FakeEncoder())


def pricing(outcome):
    return next(item for item in outcome["requirements"] if "pricing" in item["text"])


show(
    ["Run", "Pricing tier", "Credit", "Evidence", "Score", "Engine version"],
    [
        (name, pricing(outcome)["tier"], pricing(outcome)["credit"], pricing(outcome)["evidence"], outcome["overall"], outcome["engineVersion"])
        for name, outcome in [("Without a model", without_model), ("With the fake encoder", with_model)]
    ],
)

| Run | Pricing tier | Credit | Evidence | Score | Engine version |
| --- | --- | --- | --- | --- | --- |
| Without a model | unassessed | – | – | 56 | rules-1 |
| With the fake encoder | implied | 0.8 | Modelled discount pricing for 40 retail stores | 59 | rules-1+fake-1 |

## Key points

- The checklist comes from the job description, and every grade comes with its evidence where there is any.
- Using a skill in work counts for more than listing it.
- A model can only change skill and free-text requirements. Experience and education always use rules.
- All weights, credits and limits are starting values that haven't been validated against real outcomes.